# 01 · Sentence Embeddings
### Practical LLM Engineering — Module 03: Embeddings & Retrieval

> **Objectives**
> - What sentence embeddings are and how they are produced
> - The geometry of embedding space: cosine similarity, dot product, L2
> - Building embeddings from scratch with mean pooling
> - Using production embedding models (sentence-transformers, OpenAI, Cohere)
> - Dimensionality reduction and visualisation with PCA and t-SNE
> - Engineering insights: embedding size, latency, and cost

---


## 1. Overview

A **sentence embedding** is a fixed-length dense vector that represents the meaning of a piece of text in a continuous geometric space — where semantically similar texts are geometrically close.

```
"The cat sat on the mat"    → [0.12, -0.87, 0.34, ..., 0.09]  (768 dims)
"A feline rested on a rug"  → [0.11, -0.85, 0.36, ..., 0.08]  ← nearby!
"Quantum mechanics explains" → [-0.55, 0.33, -0.12, ..., 0.71] ← far away
```

Embeddings are the foundation of:
- **Semantic search** — retrieve documents by meaning, not keyword
- **RAG pipelines** — find relevant context to ground LLM answers
- **Clustering & classification** — group or classify text without labels
- **Recommendation** — find similar items based on description


## 2. Setup

In [ ]:
# Install (run once on Colab)
# !pip install sentence-transformers transformers torch numpy matplotlib scikit-learn -q

import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print("✅ Libraries ready")


In [ ]:
# ── Model-agnostic embedding client ──────────────────────────────────
from abc import ABC, abstractmethod
from dataclasses import dataclass

@dataclass
class EmbeddingResponse:
    vectors:     np.ndarray    # shape (n_texts, dim)
    model:       str
    n_tokens:    int
    latency_ms:  float
    dim:         int

    def __getitem__(self, idx): return self.vectors[idx]
    def __len__(self):          return len(self.vectors)


class BaseEmbedder(ABC):
    @abstractmethod
    def embed(self, texts: list[str]) -> EmbeddingResponse: ...
    def __call__(self, texts): return self.embed(texts)


class SentenceTransformerEmbedder(BaseEmbedder):
    """HuggingFace sentence-transformers (local, free)."""
    def __init__(self, model: str = "all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer
        self.model_name = model
        self.model = SentenceTransformer(model)
    def embed(self, texts):
        t0   = time.perf_counter()
        vecs = self.model.encode(texts, normalize_embeddings=True,
                                  show_progress_bar=False)
        ms   = (time.perf_counter() - t0) * 1000
        return EmbeddingResponse(np.array(vecs, dtype=np.float32),
                                  self.model_name,
                                  sum(len(t.split()) for t in texts),
                                  ms, vecs.shape[1])


class OpenAIEmbedder(BaseEmbedder):
    """OpenAI text-embedding-3-small (API)."""
    def __init__(self, model="text-embedding-3-small", api_key=None):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model_name = model
    def embed(self, texts):
        t0   = time.perf_counter()
        resp = self.client.embeddings.create(input=texts, model=self.model_name)
        ms   = (time.perf_counter() - t0) * 1000
        vecs = np.array([r.embedding for r in resp.data], dtype=np.float32)
        return EmbeddingResponse(vecs, self.model_name,
                                  resp.usage.total_tokens, ms, vecs.shape[1])


class CohereEmbedder(BaseEmbedder):
    """Cohere embed-english-v3 (API)."""
    def __init__(self, model="embed-english-v3.0", api_key=None):
        import cohere
        self.client = cohere.Client(api_key or os.environ.get("COHERE_API_KEY"))
        self.model_name = model
    def embed(self, texts):
        t0   = time.perf_counter()
        resp = self.client.embed(texts=texts, model=self.model_name,
                                  input_type="search_document")
        ms   = (time.perf_counter() - t0) * 1000
        vecs = np.array(resp.embeddings, dtype=np.float32)
        # Normalise
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        vecs  = vecs / np.maximum(norms, 1e-9)
        return EmbeddingResponse(vecs, self.model_name, 0, ms, vecs.shape[1])


# ── Mock embedder (no API key needed) ─────────────────────────────────
class MockEmbedder(BaseEmbedder):
    """
    Deterministic pseudo-embedder using word-hash projections.
    Produces semantically-ish consistent embeddings for demo purposes.
    NOT suitable for production — use a real model.
    """
    def __init__(self, dim: int = 64, seed: int = 42):
        self.dim  = dim
        self.seed = seed
        # Build a word projection matrix
        rng = np.random.RandomState(seed)
        self._proj = rng.randn(10_000, dim).astype(np.float32)

    def _word_hash(self, word: str) -> int:
        return int(abs(hash(word.lower())) % 10_000)

    def _text_vec(self, text: str) -> np.ndarray:
        tokens = text.lower().split()
        if not tokens:
            return np.zeros(self.dim, dtype=np.float32)
        vecs = np.array([self._proj[self._word_hash(t)] for t in tokens])
        v    = vecs.mean(axis=0)
        norm = np.linalg.norm(v)
        return v / max(norm, 1e-9)

    def embed(self, texts: list[str]) -> EmbeddingResponse:
        t0   = time.perf_counter()
        vecs = np.array([self._text_vec(t) for t in texts], dtype=np.float32)
        ms   = (time.perf_counter() - t0) * 1000
        return EmbeddingResponse(vecs, "mock-embedder",
                                  sum(len(t.split()) for t in texts),
                                  ms, self.dim)


def get_embedder(backend: str = "mock", **kwargs) -> BaseEmbedder:
    return {
        "mock":    MockEmbedder,
        "st":      SentenceTransformerEmbedder,
        "hf":      SentenceTransformerEmbedder,
        "openai":  OpenAIEmbedder,
        "cohere":  CohereEmbedder,
    }[backend.lower()](**kwargs)


# ── Activate ──────────────────────────────────────────────────────────
# Option A — no API key (default for this notebook)
embedder = get_embedder("mock", dim=64)

# Option B — local model (free, requires GPU or CPU patience)
# embedder = get_embedder("st", model="all-MiniLM-L6-v2")

# Option C — OpenAI API
# embedder = get_embedder("openai")

resp = embedder.embed(["Hello world"])
print(f"Embedder  : {resp.model}")
print(f"Dimension : {resp.dim}")
print(f"Latency   : {resp.latency_ms:.1f}ms")
print(f"Vector[0] : {resp.vectors[0][:6]}...")


## 3. Background: From Tokens to Sentence Vectors

### 3.1 How sentence embeddings are produced

Modern sentence embeddings come from encoder-only or bi-encoder transformer models:

```
Input text
    ↓  Tokenise
[CLS] The   cat   sat  [SEP]
    ↓  Transformer encoder (N layers of attention + FFN)
h_CLS  h_The  h_cat  h_sat
    ↓  Pooling strategy
sentence_vector ∈ ℝ^d
```

**Three common pooling strategies:**

| Strategy | Formula | Notes |
|---|---|---|
| CLS pooling | $\mathbf{v} = h_{\text{CLS}}$ | Used by BERT-style models |
| Mean pooling | $\mathbf{v} = \frac{1}{T}\sum_t h_t$ | More robust, used by sentence-transformers |
| Max pooling | $\mathbf{v}_i = \max_t h_{t,i}$ | Captures peak activation per dimension |

---

### 3.2 Similarity measures

Given two embedding vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$:

**Cosine similarity** (angle between vectors):
$$
\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|} \in [-1, 1]
$$

**Dot product** (magnitude + angle):
$$
\text{dot}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i
$$

**L2 distance** (Euclidean):
$$
\text{L2}(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|_2 = \sqrt{\sum_i (a_i - b_i)^2}
$$

**Engineering note:** If embeddings are L2-normalised ($\|\mathbf{v}\|=1$), cosine similarity and dot product are equivalent. Most production embedders produce normalised vectors for this reason.


## 4. Building Mean-Pooled Embeddings from Scratch

In [ ]:
import torch
import torch.nn.functional as F

def mean_pool_embeddings(model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
                          texts: list[str] = None) -> np.ndarray:
    """
    Produce sentence embeddings via mean pooling over token hidden states.
    Equivalent to SentenceTransformer's default pooling.
    """
    # Uncomment to run with a real model:
    # from transformers import AutoTokenizer, AutoModel
    # tokenizer = AutoTokenizer.from_pretrained(model_name)
    # model     = AutoModel.from_pretrained(model_name)
    # model.eval()
    #
    # with torch.no_grad():
    #     encoded = tokenizer(texts, padding=True, truncation=True,
    #                          return_tensors="pt", max_length=512)
    #     outputs = model(**encoded)
    #     hidden  = outputs.last_hidden_state          # (B, T, D)
    #     mask    = encoded["attention_mask"]          # (B, T)
    #     # Mean pool: sum(hidden * mask) / sum(mask)
    #     expanded = mask.unsqueeze(-1).float()
    #     summed   = (hidden * expanded).sum(dim=1)
    #     counts   = expanded.sum(dim=1)
    #     pooled   = summed / counts.clamp(min=1e-9)
    #     # L2-normalise
    #     normed   = F.normalize(pooled, p=2, dim=1)
    #     return normed.numpy()

    # Simulate with random vectors for notebook execution
    B, D = len(texts or [""]), 384
    torch.manual_seed(sum(hash(t) for t in (texts or [""])) % 1000)
    vecs  = torch.randn(B, D)
    normed = F.normalize(vecs, p=2, dim=1)
    return normed.numpy()


# ── Demonstrate mean vs CLS pooling difference ────────────────────────
texts = [
    "The capital of France is Paris.",
    "Paris is the capital city of France.",
    "Berlin is the capital of Germany.",
    "Machine learning models learn from data.",
]

# Simulate both pooling strategies
np.random.seed(0)
mean_vecs = mean_pool_embeddings(texts=texts)
# Simulate CLS (slightly different due to CLS token attention)
cls_vecs  = mean_pool_embeddings(texts=texts)
cls_vecs += np.random.randn(*cls_vecs.shape) * 0.05   # small perturbation
cls_vecs /= np.linalg.norm(cls_vecs, axis=1, keepdims=True)


def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


print("Pairwise cosine similarities (mean pooling):")
print(f"  {'Pair':<50} {'Sim':>6}")
print("-" * 58)
pairs = [(0,1,"Paris sentences (paraphrase)"),
         (0,2,"Paris vs Berlin (same type, different)"),
         (0,3,"Paris vs ML (unrelated)")]
for i, j, label in pairs:
    s = cosine_sim(mean_vecs[i], mean_vecs[j])
    print(f"  {label:<50} {s:>6.3f}")


## 5. The Geometry of Embedding Space

In [ ]:
# ── Similarity metrics comparison ────────────────────────────────────
def compute_similarities(vecs: np.ndarray) -> dict:
    """Compute cosine, dot product, and L2 similarity matrices."""
    n = len(vecs)
    cos  = np.zeros((n, n))
    dot  = np.zeros((n, n))
    l2   = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            cos[i,j] = cosine_sim(vecs[i], vecs[j])
            dot[i,j] = np.dot(vecs[i], vecs[j])
            l2[i,j]  = np.linalg.norm(vecs[i] - vecs[j])
    return {"cosine": cos, "dot": dot, "l2": l2}


# ── Corpus for geometry exploration ──────────────────────────────────
CORPUS = {
    "animals": [
        "The cat climbed the tree",
        "A dog chased the ball across the yard",
        "The lion hunted prey on the savanna",
        "Parrots can mimic human speech",
    ],
    "tech": [
        "Neural networks learn by backpropagation",
        "Transformer models use attention mechanisms",
        "Deep learning requires large datasets",
        "GPUs accelerate matrix multiplications",
    ],
    "cooking": [
        "Sauté the onions until golden brown",
        "Add garlic and stir for two minutes",
        "Season with salt, pepper, and herbs",
        "Simmer the sauce on low heat",
    ],
}

all_texts  = [t for cat in CORPUS.values() for t in cat]
all_labels = [cat for cat, texts in CORPUS.items() for _ in texts]

resp = embedder.embed(all_texts)
vecs = resp.vectors

sims = compute_similarities(vecs)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Similarity matrices: cosine · dot product · L2 distance", fontsize=11)

short_labels = [f"{l[:3]}{i}" for i, l in enumerate(all_labels)]
for ax, (metric, mat) in zip(axes, sims.items()):
    cmap = "RdYlGn" if metric != "l2" else "RdYlGn_r"
    im   = ax.imshow(mat, cmap=cmap, vmin=mat.min(), vmax=mat.max())
    ax.set_xticks(range(len(short_labels))); ax.set_xticklabels(short_labels, rotation=90, fontsize=7)
    ax.set_yticks(range(len(short_labels))); ax.set_yticklabels(short_labels, fontsize=7)
    ax.set_title(metric)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()
print("Block structure along the diagonal = within-category similarity is higher ✅")


## 6. Visualising Embedding Space: PCA and t-SNE

In [ ]:
# ── PCA: linear projection ────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
vecs_2d_pca = pca.fit_transform(vecs)
explained   = pca.explained_variance_ratio_

# ── t-SNE: non-linear projection ─────────────────────────────────────
# Use PCA init for stability
tsne = TSNE(n_components=2, perplexity=min(5, len(vecs)-1),
            random_state=42, init="pca", learning_rate="auto")
vecs_2d_tsne = tsne.fit_transform(vecs)

# ── Plot ──────────────────────────────────────────────────────────────
CATEGORY_COLORS = {"animals": "#e74c3c", "tech": "#3498db", "cooking": "#2ecc71"}
colors = [CATEGORY_COLORS[l] for l in all_labels]
markers = {"animals": "o", "tech": "s", "cooking": "^"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Embedding space visualisation: 12 sentences, 3 categories", fontsize=11)

for ax, (proj_name, proj_vecs, extra) in zip(axes, [
    ("PCA",  vecs_2d_pca,  f"({explained[0]:.0%} + {explained[1]:.0%} variance)"),
    ("t-SNE", vecs_2d_tsne, "(non-linear, perplexity=5)"),
]):
    for cat, color in CATEGORY_COLORS.items():
        idxs  = [i for i, l in enumerate(all_labels) if l == cat]
        xvals = proj_vecs[idxs, 0]
        yvals = proj_vecs[idxs, 1]
        ax.scatter(xvals, yvals, c=color, marker=markers[cat],
                   s=120, label=cat, edgecolors="white", linewidths=0.8, zorder=3)
        for idx in idxs:
            ax.annotate(all_texts[idx][:22], proj_vecs[idx],
                        fontsize=6.5, alpha=0.7,
                        xytext=(3, 3), textcoords="offset points")
    ax.set_title(f"{proj_name} {extra}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
    ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")

plt.tight_layout()
plt.show()


## 7. Production Embedding Model Landscape

### 7.1 Popular models and their tradeoffs

| Model | Dim | Max tokens | Speed | Quality | Cost |
|---|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 256 | ⚡⚡⚡ | ★★★ | Free (local) |
| `all-mpnet-base-v2` | 768 | 514 | ⚡⚡ | ★★★★ | Free (local) |
| `bge-large-en-v1.5` | 1024 | 512 | ⚡ | ★★★★★ | Free (local) |
| `text-embedding-3-small` | 1536 | 8191 | ⚡⚡⚡ | ★★★★ | $0.02/1M tokens |
| `text-embedding-3-large` | 3072 | 8191 | ⚡⚡ | ★★★★★ | $0.13/1M tokens |
| `embed-english-v3.0` (Cohere) | 1024 | 512 | ⚡⚡ | ★★★★★ | $0.10/1M tokens |
| `voyage-3` (Voyage AI) | 1024 | 32000 | ⚡⚡ | ★★★★★ | $0.06/1M tokens |

### 7.2 Choosing a model

```
Latency-critical, on-device:    all-MiniLM-L6-v2 (22M params, 80MB)
Best open-source quality:       bge-large-en-v1.5 or e5-large-v2
Cloud API, general purpose:     text-embedding-3-small
Highest quality, multilingual:  text-embedding-3-large or voyage-3
```


In [ ]:
# ── Model comparison experiment ────────────────────────────────────────
# Simulate comparing embedding model quality on a small retrieval task

# Query + relevant doc + irrelevant doc
RETRIEVAL_TEST = [
    {
        "query":     "How do transformers use attention?",
        "relevant":  "Transformers compute attention weights between all token pairs.",
        "irrelevant":"The chef prepared a delicious pasta dish.",
    },
    {
        "query":     "What causes inflation?",
        "relevant":  "Inflation occurs when the money supply grows faster than output.",
        "irrelevant":"The hiking trail winds through dense forest.",
    },
    {
        "query":     "How does gradient descent work?",
        "relevant":  "Gradient descent minimises loss by moving in the negative gradient direction.",
        "irrelevant":"Butterflies migrate thousands of kilometres each season.",
    },
]

# Simulate three model "quality tiers" with different noise levels
def simulate_embedder(quality: float, dim: int = 64, seed: int = 0):
    """Higher quality = better separation between relevant/irrelevant."""
    np.random.seed(seed)
    results = []
    for item in RETRIEVAL_TEST:
        q_vec = np.random.randn(dim)
        # Relevant: add correlated signal proportional to quality
        r_vec = q_vec * quality + np.random.randn(dim) * (1 - quality)
        i_vec = np.random.randn(dim)    # irrelevant: random
        for v in [q_vec, r_vec, i_vec]:
            v /= np.linalg.norm(v)
        sim_rel = float(np.dot(q_vec, r_vec))
        sim_irr = float(np.dot(q_vec, i_vec))
        results.append({"sim_rel": sim_rel, "sim_irr": sim_irr,
                         "correct": sim_rel > sim_irr})
    return results


models_sim = {
    "all-MiniLM-L6-v2":        simulate_embedder(0.65, seed=1),
    "all-mpnet-base-v2":        simulate_embedder(0.75, seed=2),
    "bge-large-en-v1.5":        simulate_embedder(0.85, seed=3),
    "text-embedding-3-small":   simulate_embedder(0.80, seed=4),
    "text-embedding-3-large":   simulate_embedder(0.90, seed=5),
}

print("Retrieval accuracy on 3-item test set:")
print(f"{'Model':<28} {'Acc':>6}  {'Avg sim_rel':>12}  {'Avg sim_irr':>12}  {'Gap':>8}")
print("-" * 72)
for model_name, results in models_sim.items():
    acc      = sum(r["correct"] for r in results) / len(results)
    sim_rel  = np.mean([r["sim_rel"] for r in results])
    sim_irr  = np.mean([r["sim_irr"] for r in results])
    gap      = sim_rel - sim_irr
    print(f"{model_name:<28} {acc:>6.0%}  {sim_rel:>12.3f}  {sim_irr:>12.3f}  {gap:>8.3f}")


## 8. Engineering: Embedding Dimensionality

### 8.1 Dimension vs quality vs storage

Larger embedding dimensions capture more nuance but increase:
- **Memory** — $d \times 4$ bytes per vector (float32)
- **Compute** — similarity search scales with $d$
- **Index size** — larger ANN indices

### 8.2 Matryoshka Representation Learning (MRL)

OpenAI's `text-embedding-3` models support **truncating** embeddings to smaller dimensions without significant quality loss — a technique called MRL (Kusupati et al., 2022):

$$
\text{quality}(d) \approx \text{quality}(D) \cdot \left(1 - e^{-\lambda d / D}\right)
$$

This lets you trade off quality for storage/speed at query time.


In [ ]:
# ── Dimension vs quality tradeoff ────────────────────────────────────
dims = [32, 64, 128, 256, 384, 512, 768, 1024, 1536, 3072]

# Simulate quality (MTEB score proxy) and storage
def quality_at_dim(d, d_max=3072, base_quality=0.65):
    """Simulated quality curve following MRL-like behaviour."""
    return base_quality + (1 - base_quality) * (1 - np.exp(-3 * d / d_max))

qualities = [quality_at_dim(d) for d in dims]
storage_mb_per_1M = [d * 4 / 1e6 * 1_000_000 for d in dims]  # MB per 1M vecs

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Embedding Dimensionality Tradeoffs", fontsize=11)

ax1.plot(dims, qualities, "o-", color="#3498db", lw=2)
ax1.set_xlabel("Embedding dimension"); ax1.set_ylabel("Retrieval quality (proxy)")
ax1.set_title("Quality vs dimension"); ax1.grid(alpha=0.3)
ax1.set_xscale("log")

ax2.plot(dims, storage_mb_per_1M, "s-", color="#e67e22", lw=2)
ax2.set_xlabel("Embedding dimension"); ax2.set_ylabel("Storage (MB) per 1M vectors")
ax2.set_title("Storage cost"); ax2.grid(alpha=0.3)
ax2.set_xscale("log")

# Efficiency: quality per MB
eff = [q / (s/1000) for q, s in zip(qualities, storage_mb_per_1M)]
ax3.plot(dims, eff, "^-", color="#2ecc71", lw=2)
ax3.set_xlabel("Embedding dimension"); ax3.set_ylabel("Quality per GB")
ax3.set_title("Storage efficiency"); ax3.grid(alpha=0.3)
ax3.set_xscale("log")

plt.tight_layout()
plt.show()

print("Storage requirements for 1 million float32 vectors:")
for d in [128, 384, 768, 1536, 3072]:
    mb = d * 4 * 1_000_000 / 1e6
    print(f"  dim={d:<5}  {mb:>6.0f} MB  ({mb/1024:.2f} GB)")


## 9. Efficient Batch Embedding

In [ ]:
# ── Batching strategy ────────────────────────────────────────────────
def embed_in_batches(embedder: BaseEmbedder, texts: list[str],
                      batch_size: int = 64,
                      show_progress: bool = True) -> np.ndarray:
    """
    Embed a large corpus in fixed-size batches.
    Handles rate limits, memory pressure, and progress logging.
    """
    all_vecs = []
    n_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch  = texts[i : i + batch_size]
        resp   = embedder.embed(batch)
        all_vecs.append(resp.vectors)
        if show_progress and ((i // batch_size + 1) % max(1, n_batches//5) == 0
                               or i + batch_size >= len(texts)):
            print(f"  Embedded {min(i+batch_size, len(texts))}/{len(texts)} texts...")

    return np.vstack(all_vecs)


# ── Benchmark batch sizes ─────────────────────────────────────────────
corpus_large = [f"Sample document number {i} with some content about topic {i%10}."
                for i in range(200)]

print("Batching strategy benchmark (200 texts):")
print(f"{'Batch size':>12} {'n_batches':>10} {'Time (ms)':>10} {'Throughput (texts/s)':>22}")
print("-" * 58)
for bs in [1, 8, 32, 64, 128, 200]:
    t0    = time.perf_counter()
    vecs  = embed_in_batches(embedder, corpus_large, batch_size=bs, show_progress=False)
    ms    = (time.perf_counter() - t0) * 1000
    n_bat = (len(corpus_large) + bs - 1) // bs
    thr   = len(corpus_large) / (ms / 1000)
    print(f"{bs:>12} {n_bat:>10} {ms:>10.1f} {thr:>22.0f}")


## 10. Engineering Insights

### 10.1 Embedding pipeline cost

| Corpus size | Model | Tokens | API cost |
|---|---|---|---|
| 10K docs × 500 tokens | text-embedding-3-small | 5M | $0.10 |
| 10K docs × 500 tokens | text-embedding-3-large | 5M | $0.65 |
| 1M docs × 500 tokens | text-embedding-3-small | 500M | $10.00 |
| 1M docs × 500 tokens | Local (MiniLM) | 500M | $0 + GPU time |

### 10.2 Embedding cache strategy

Embeddings are **deterministic** for a fixed model — always cache them:

```python
import hashlib, pickle, os

def cached_embed(embedder, text: str, cache_dir: str = ".embed_cache") -> np.ndarray:
    key  = hashlib.md5(f"{embedder.model_name}:{text}".encode()).hexdigest()
    path = os.path.join(cache_dir, f"{key}.npy")
    if os.path.exists(path):
        return np.load(path)
    vec = embedder.embed([text]).vectors[0]
    os.makedirs(cache_dir, exist_ok=True)
    np.save(path, vec)
    return vec
```

### 10.3 When to re-embed

| Event | Action |
|---|---|
| Document content changes | Re-embed that document only |
| Embedding model upgraded | Re-embed entire corpus |
| Dimension truncation (MRL) | Re-truncate in-place (no re-encode) |
| Query at inference time | Always embed fresh (no cache) |


## 11. Exercises

1. **Pooling comparison:** Use a real `transformers` model to extract CLS, mean-pool, and max-pool embeddings for 10 sentence pairs. Compute pairwise cosine similarity for each pooling strategy. Which correlates best with human-judged semantic similarity?

2. **Dimension ablation:** Using a real `sentence-transformers` model with 768-dim output, truncate to [64, 128, 256, 384, 768] dimensions and measure retrieval accuracy (recall@5) on a 50-query test set. At what dimension does quality plateau?

3. **Cross-model alignment:** Embed the same 20 sentences with two different models (e.g. MiniLM and MPNet). Compute the similarity matrices for both. How correlated are the resulting orderings (Spearman ρ)? What does this mean for switching embedding models in production?

4. **Caching benchmark:** Implement the `cached_embed` function above and measure the speedup on 1000 repeated embeddings. How much does caching matter for a query-heavy system?

5. **MRL truncation:** For `text-embedding-3-small` (1536-dim), truncate to [64, 128, 256, 512, 1024, 1536] and evaluate on 20 retrieval queries. At what truncation does performance drop more than 5%?

---

## 12. Key Takeaways

| Concept | Summary |
|---|---|
| **Sentence embedding** | Fixed-length dense vector encoding semantic meaning |
| **Mean pooling** | Average of token hidden states — robust and widely used |
| **Cosine similarity** | Angle-based similarity; equivalent to dot product on normalised vectors |
| **Normalisation** | L2-normalise all vectors — enables fast dot product as cosine sim |
| **PCA / t-SNE** | Linear / non-linear projections for visualising embedding clusters |
| **Dimensionality** | Larger dim = more quality; 384–768 is sweet spot for most tasks |
| **MRL** | Truncate at query time to save compute without re-encoding |
| **Caching** | Embeddings are deterministic — always cache document embeddings |
| **Model choice** | Local MiniLM for speed; BGE-large or voyage-3 for highest quality |
